# Drought Pulse — SWI vs ETA

Dual-panel animation of Soil Water Index (daily, 12.5 km) and
Actual Evapotranspiration (10-daily / dekadal, 300 m) over
Central Europe, highlighting the 2022 drought.

**Note:** SWI and ETA have different spatial resolutions (12.5 km
vs 300 m) and temporal cadences (daily vs dekadal).


In [ ]:
from functools import partial
from pathlib import Path

from rs_tools.config import BoundingBox
from rs_tools.datasets.loader import load_dataset, load_passes_from_disk
from rs_tools.visualization.animation import save_timeseries_gif_lazy
from rs_tools.visualization.frames import make_dual_panel_composite
from rs_tools.visualization.clms_colormaps import CLMS_SWI, SWI_VMIN, SWI_VMAX, CLMS_ETA, ETA_VMIN, ETA_VMAX

In [ ]:
bbox = BoundingBox(west=-5, south=42, east=20, north=55)
DATA_DIR = "/home/bekaertd/RS_applications/Applications/CGOPS/drought_pulse"
gif_dir = Path('output/gifs')
gif_dir.mkdir(parents=True, exist_ok=True)

# Which dekads to include: [1], [2], [3], [1,2], etc.  None = all dekads.
DEKADS = None

## Load SWI (daily, 12.5 km) and ETA (dekadal, 300 m)

In [ ]:
swi_items = load_dataset(
    "CLMS_SWI_V4", bbox=bbox,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=150, output_dir=f"{DATA_DIR}/swi", dekads=DEKADS,
)
print(f"SWI: {len(swi_items)} items")

In [ ]:
eta_items = load_dataset(
    "CLMS_ETA_V1", bbox=bbox,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=150, output_dir=f"{DATA_DIR}/eta", dekads=DEKADS,
)
print(f"ETA: {len(eta_items)} dekads")

In [ ]:
# Reload as lightweight metadata references (no pixel data in RAM)
swi_items = load_passes_from_disk(f"{DATA_DIR}/swi", dekads=DEKADS)
eta_items = load_passes_from_disk(f"{DATA_DIR}/eta", dekads=DEKADS)
n = min(len(swi_items), len(eta_items))
swi_items, eta_items = swi_items[:n], eta_items[:n]
print(f"SWI: {n} items  ETA: {n} dekads")

## Dual-panel GIF (lazy — one frame at a time)

In [ ]:
dual_composite = partial(
    make_dual_panel_composite,
    left_cmap=CLMS_SWI, right_cmap=CLMS_ETA,
    left_vmin=SWI_VMIN, left_vmax=SWI_VMAX,
    right_vmin=ETA_VMIN, right_vmax=ETA_VMAX,
    left_label="SWI", right_label="ETA",
)

def _composite(pair):
    left, right = pair
    return dual_composite(left, right)

gif_path = save_timeseries_gif_lazy(
    zip(swi_items, eta_items),
    gif_dir / "drought_pulse_swi_eta.gif",
    composite_fn=_composite,
    title="Drought Pulse — SWI vs ETA",
    fps=4,
    figsize=(16, 8),
)
print(f"Saved: {gif_path}")